<a href="https://colab.research.google.com/github/tanercc/python-colab/blob/dev/tjk_to_Predict_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q mysql-connector-python
!pip install pandas tensorflow matplotlib

In [ ]:
# model history graph
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title("Training Progress")
plt.grid(True)
plt.show()

In [ ]:
# model history graph
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title("Training Progress")
plt.grid(True)
plt.show()

Get TJK DB

In [ ]:
import mysql.connector as connection
import pandas as pd
try:
    mydb = connection.connect(host="taner.web.tr", database = 'tanerweb_tjk',user="tanerweb_tjk", passwd="Ka9jVjRJRtRW",use_pure=True)
    query = """
SELECT
`atlar`.`KOD` AS `code`,
`atlar`.`ADKUCUK` AS `name`,
`kosular`.`GRUP_EN` AS `group`,
`atlar`.`ANNE` AS `mother`,
`atlar`.`BABA` AS `father`,
`kosular`.`TARIH` AS `date`,
`hipodrom`.`AD` AS `hipname`,
`hipodrom`.`KOD` AS `hipcode`,
`kosular`.`ActiveClass` AS `surface`,
COALESCE(CASE WHEN `kosular`.`PIST` = 'cim' THEN `hava`.`CIM_EN` ELSE `hava`.`KUM_EN` END, 'Normal') AS `ground`,
`kosular`.`RACENO` AS `raceno`,
`atlar`.`KILO` + `atlar`.`FAZLAKILO` AS `weight`,
`atlar`.`JOKEYADI` AS `jockey`,
`kosular`.`MESAFE` AS `metre`,
`atlar`.`DERECE` AS `time`,
COALESCE(`atlar`.`SONUC`, (SELECT COUNT(NO) FROM `atlar` AS `atsay` WHERE `atlar`.`TARIHKOD`=`atsay`.`TARIHKOD` AND `atlar`.`HIPODROMKOD`=`atsay`.`HIPODROMKOD` AND `atlar`.`KOSUNO`=`atsay`.`KOSUNO`)) AS `pos`
FROM `atlar`
LEFT JOIN `hava` ON (`atlar`.`TARIHKOD` = `hava`.`TARIHKOD` AND `atlar`.`HIPODROMKOD` = `hava`.`HIPODROMKOD`)
LEFT JOIN `kosular` ON (`kosular`.`TARIHKOD` = `atlar`.`TARIHKOD` AND `atlar`.`HIPODROMKOD` = `kosular`.`HIPODROMKOD` AND `atlar`.`KOSUNO` = `kosular`.`NO`)
LEFT JOIN `hipodrom` ON `hipodrom`.`KOD` = `atlar`.`HIPODROMKOD`
WHERE `atlar`.`TARIHKOD` > 0
AND `atlar`.`KILO` > 0
AND `atlar`.`KOSMAZ` = 0
AND `atlar`.`GECCIKIS_BOY` = ''
AND `atlar`.`START` > 0
AND `atlar`.`DERECE` > 0
AND `atlar`.`SONUC` > 0
ORDER BY `atlar`.`TARIHKOD` DESC,`atlar`.`HIPODROMKOD`,`atlar`.`KOSUNO`
    """
    #AND `atlar`.`HIPODROMKOD` = 1
    df = pd.read_sql(query,mydb)
    mydb.close() #close the connection
except Exception as e:
    mydb.close()
    print(str(e))

In [ ]:
df.head(10)

In [ ]:
#df.to_csv("data-output.csv")

In [ ]:
import pandas as pd

# STEP 1: Calculate breed scores for father
father_stats = df.groupby("father").agg(
    race_count=("pos", "count"),
    avg_pos=("pos", "mean"),
    avg_time=("time", "mean"),
    win_count=("pos", lambda x: (x == 1).sum())
)
father_stats["win_rate"] = father_stats["win_count"] / father_stats["race_count"]
father_stats["breed_score"] = father_stats["avg_pos"] * 0.5 + father_stats["avg_time"] * 0.5

# STEP 2: Calculate breed scores for mother
mother_stats = df.groupby("mother").agg(
    race_count=("pos", "count"),
    avg_pos=("pos", "mean"),
    avg_time=("time", "mean"),
    win_count=("pos", lambda x: (x == 1).sum())
)
mother_stats["win_rate"] = mother_stats["win_count"] / mother_stats["race_count"]
mother_stats["breed_score"] = mother_stats["avg_pos"] * 0.5 + mother_stats["avg_time"] * 0.5

# STEP 3: Map scores back to original dataframe
df["father_breed_score"] = df["father"].map(father_stats["breed_score"])
df["father_win_rate"] = df["father"].map(father_stats["win_rate"])
df["mother_breed_score"] = df["mother"].map(mother_stats["breed_score"])
df["mother_win_rate"] = df["mother"].map(mother_stats["win_rate"])

# DONE! Preview
print(df[[
    "name", "mother", "father",
    "father_breed_score", "father_win_rate", "mother_breed_score", "mother_win_rate"
]])

# Show result
df.head()

In [ ]:
import pandas as pd

# Helper: create a breed_score column per row
df["breed_score_row"] = df["pos"] * 0.5 + df["time"] * 0.5

# STEP 1 — MOTHER-based sibling stats
mother_stats = df.groupby("mother").agg(
    sibling_mother_score=("breed_score_row", "mean"),
    sibling_mother_win_rate=("pos", lambda x: (x == 1).sum() / len(x))
)

# STEP 2 — FATHER-based sibling stats
father_stats = df.groupby("father").agg(
    sibling_father_score=("breed_score_row", "mean"),
    sibling_father_win_rate=("pos", lambda x: (x == 1).sum() / len(x))
)

# STEP 3 — Map them to the main DataFrame
df["sibling_mother_score"] = df["mother"].map(mother_stats["sibling_mother_score"])
df["sibling_mother_win_rate"] = df["mother"].map(mother_stats["sibling_mother_win_rate"])
df["sibling_father_score"] = df["father"].map(father_stats["sibling_father_score"])
df["sibling_father_win_rate"] = df["father"].map(father_stats["sibling_father_win_rate"])

# STEP 4 — Combine scores (optional, average of both sides)
df["sibling_breed_score"] = (
    df["sibling_mother_score"] + df["sibling_father_score"]
) / 2

df["sibling_win_rate"] = (
    df["sibling_mother_win_rate"] + df["sibling_father_win_rate"]
) / 2

# DONE! Preview
print(df[[
    "name", "mother", "father",
    "sibling_breed_score", "sibling_win_rate"
]])

# Show result
df.head()

In [ ]:
# Combine all breed scores into one
df["combined_breed_score"] = df[[
    "father_breed_score",
    "mother_breed_score",
    "sibling_breed_score"
]].mean(axis=1)

# Optional: round for readability
df["combined_breed_score"] = df["combined_breed_score"].round(2)

# Preview
print(df[[
    "name", "father_breed_score", "mother_breed_score", "sibling_breed_score", "combined_breed_score"
]])

# Show result
df.head()

In [ ]:
# STEP 1 — Calculate jockey performance stats
jockey_stats = df.groupby("jockey").agg(
    jockey_avg_pos=("pos", "mean"),
    jockey_avg_time=("time", "mean"),
    jockey_win_count=("pos", lambda x: (x == 1).sum()),
    race_count=("pos", "count")
)

# STEP 2 — Add win rate and jockey score
jockey_stats["jockey_win_rate"] = jockey_stats["jockey_win_count"] / jockey_stats["race_count"]
jockey_stats["jockey_score"] = (
    jockey_stats["jockey_avg_pos"] * 0.5 + jockey_stats["jockey_avg_time"] * 0.5
)

# STEP 3 — Map stats back to DataFrame
df["jockey_score"] = df["jockey"].map(jockey_stats["jockey_score"])
df["jockey_win_rate"] = df["jockey"].map(jockey_stats["jockey_win_rate"])

# Optional: round for readability
df["jockey_score"] = df["jockey_score"].round(2)
df["jockey_win_rate"] = df["jockey_win_rate"].round(2)

#Save
df.to_csv("data-output.csv", index=False)

# DONE! Preview
print(df[[
    "name", "jockey", "jockey_score", "jockey_win_rate"
]])

# Show result
df.head()

In [ ]:
# Load data
#df = pd.read_csv("data-output.csv")

# Drop unnecessary columns (names, codes, date, etc.)
df = df.drop(columns=['father_breed_score', 'father_win_rate', 'mother_breed_score', 'mother_win_rate', 'breed_score_row', 'sibling_mother_score', 'sibling_mother_win_rate', 'sibling_father_score', 'sibling_father_win_rate', 'sibling_breed_score', 'sibling_win_rate', 'jockey_win_rate'])

# Preview
df.head()


In [ ]:
#Create new table with selected columns
selected_columns = [
    "name",
    "group",
    "surface",
    "ground",
    "combined_breed_score",
    "jockey_score",
    "weight",
    "metre",
    "time"
]

df = df[selected_columns]
print(df.size)
df.head()

In [ ]:
random_rows = df.sample(n=50)
random_rows.head()

#count = df['surface'].nunique()
#print(count)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError
from sklearn.metrics import mean_absolute_error

# Fix numerics
numeric_cols = ["combined_breed_score", "jockey_score", 'weight', 'metre', 'time']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
df = df.dropna(subset=numeric_cols)

# Categorical columns
cat_cols = ['name', 'group', 'surface', 'ground']
num_cols = ['combined_breed_score', 'jockey_score', 'weight', 'metre']

# Normalize columns
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# Features and label
X = df[cat_cols + num_cols]
y = df["time"]

#possibilities

# Preprocessing: OneHot for categoricals, Scale numerics
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ("num", StandardScaler(), num_cols)
])

X_processed = preprocessor.fit_transform(X)

# Initialize KFold cross-validation
kf = KFold(n_splits=2, shuffle=True, random_state=42)

# Initialize list to store MAE for each fold
mae_scores = []

model = None
history = None

# K-fold cross-validation
for train_index, val_index in kf.split(X_processed):
    X_train, X_val = X_processed[train_index], X_processed[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Build Neural Network model
    if model is None:
        model = Sequential([
            Input(shape=(X_train.shape[1],)),
            Dense(32, activation='relu'),
            BatchNormalization(),
            Dense(1)
        ])

        model.compile(
            optimizer=Adam(learning_rate=0.0001),
            loss=MeanSquaredError(),
            metrics=['mae']
        )

        model_save = ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_mae', mode='min')

    #early_stop = EarlyStopping(patience=10, restore_best_weights=True)

    # Train the model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=32,
        # callbacks=[early_stop],
        callbacks=[model_save],
        verbose=1
    )

    # Evaluate the model on validation set and store MAE
    y_val_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_val_pred)
    mae_scores.append(mae)
    print(f"Fold MAE: {mae:.2f} seconds")

# Calculate the average MAE across all folds
avg_mae = np.mean(mae_scores)
print(f"\nAverage MAE across all folds: {avg_mae:.2f} seconds")


In [ ]:
# model history graph
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title("Training Progress")
plt.grid(True)
plt.show()

In [ ]:
# Save the model (full model: architecture + weights + optimizer)
model.save("~/race_time_model-v2.keras")
print("✅ Model saved as race_time_model.keras")

In [ ]:
import joblib

# Save the preprocessor pipeline
joblib.dump(preprocessor, "preprocessor-v2.pkl")
print("✅ Preprocessor saved as preprocessor.pkl")

In [ ]:
# Create a new input DataFrame
new_data = pd.DataFrame([{
    "name": "Beautiful Mina",
    "group": "3 Years Old Thoroughbreds",
    "mother": "Mİ BOMBON",
    "father": "SUPER SAVER (USA)",
    "surface": "sand",
    "ground": "Good Going",
    "weight": 57.0,
    "jockey": "AHMET ÇELİK",
    "metre": 1400
}])

# Preprocess and predict
new_processed = preprocessor.transform(new_data)
predicted_time = model.predict(new_processed)
print("Predicted Time:", round(predicted_time[0][0], 2), "seconds")


In [ ]:
from tensorflow.keras.models import load_model
import joblib
import pandas as pd

# Load the model and preprocessor
model_new = load_model("race_time_model-v2.keras")
preprocessor_new = joblib.load("preprocessor-v2.pkl")

# Example: New input
new_data = pd.DataFrame([{
    "name": "Beautiful Mina",
    "group": "3 Years Old Thoroughbreds",
    "mother": "Mİ BOMBON",
    "father": "SUPER SAVER (USA)",
    "surface": "sand",
    "ground": "Good Going",
    "weight": 57.0,
    "jockey": "AHMET ÇELİK",
    "metre": 1400
}])

# Transform and predict
new_processed = preprocessor_new.transform(new_data)
predicted_time = model_new.predict(new_processed)

print("🏁 Predicted Time:", round(predicted_time[0][0], 2), "seconds")

In [ ]:
#Calculate the nonlineer effect of metre on time
import matplotlib.pyplot as plt
import seaborn as sns

sns.scatterplot(x=df["metre"], y=df["time"])
plt.title("Metre vs Time")
plt.xlabel("Distance (metre)")
plt.ylabel("Race Time (seconds)")
plt.grid(True)
plt.show()

In [ ]:
#Plot predicted vs actual times:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

y_pred = model.predict(X_test)

plt.scatter(y_test, y_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Prediction Accuracy")
plt.grid(True)

In [ ]:
query = "SELECT * FROM `tahminler` WHERE `racedate` = 20250410"
try:
    mydb = connection.connect(host="taner.web.tr", database = 'tanerweb_tjk',user="tanerweb_tjk", passwd="Ka9jVjRJRtRW",use_pure=True)
    df = pd.read_sql(query,mydb)
    mydb.close() #close the connection
except Exception as e:
    mydb.close()
    print(str(e))

df["predicted_time"] = ""

# Fix numerics
numeric_cols = ['weight', 'metre', 'time']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
df = df.dropna(subset=numeric_cols)

# Categorical columns
cat_cols = ['name', 'group', 'mother', 'father', 'surface', 'ground', 'jockey']
num_cols = ['weight', 'metre']

# Features and label
pX = df[cat_cols + num_cols]

df.head()

In [ ]:
# Load the model and preprocessor
model_new = load_model("race_time_model-v2.keras")
preprocessor_new = joblib.load("preprocessor-v2.pkl")

# Transform and predict
for index, row in pX.iterrows():
    new_data = pd.DataFrame([row])
    new_processed = preprocessor_new.transform(new_data)
    predicted_time = model_new.predict(new_processed)
    df.at[index, "predicted_time"] = round(predicted_time[0][0], 2)

df.head()

In [ ]:
df